# ViT-Small: Append Folds 3 and 4

Run this notebook after attaching the previous output dataset for `vit_small`. It merges the old folds into `/kaggle/working`, trains folds 3 and 4, then leaves one output dataset containing folds 0-4.

In [ ]:
%pip install -q timm pydicom captum grad-cam scikit-image scipy seaborn

from pathlib import Path
import json, os, shutil, subprocess, sys, time
import pandas as pd

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
PY = sys.executable

# Attach this model's previous output dataset, then run the notebook again.
SELECTED_MODEL = "vit_small"
FOLDS_TO_TRAIN = [3, 4]
EVAL_FOLDS = [0, 1, 2, 3, 4]
XAI_FOLDS = [0, 1, 2, 3, 4]  # set to [3, 4] if you only want the append pass
MAX_SAMPLES = 300
RUN_CONSENSUS = True
SAVE_MAPS = True
PARALLEL_FOLDS_IF_2GPU = True

METHODS = [
    "gradcam",
    "gradcam++",
    "integrated_gradients",
    "gradient_shap",
    "occlusion",
    "guided_backprop",
]

CFG_NAME = {
    "convnext_blackbox": "configs/baselines/convnext_blackbox.yaml",
    "cbm_nonleaky": "configs/baselines/cbm_nonleaky.yaml",
    "cbm_leaky": "configs/baselines/cbm_leaky.yaml",
    "resnet50": "configs/baselines/resnet50.yaml",
    "densenet121": "configs/baselines/densenet121.yaml",
    "efficientnet_b4": "configs/baselines/efficientnet_b4.yaml",
    "vit_small": "configs/baselines/vit_small.yaml",
    "deit_small": "configs/baselines/deit_small.yaml",
}
EXP = {
    "convnext_blackbox": "baseline_convnext_tiny_blackbox",
    "cbm_nonleaky": "cbm_nonleaky_7concepts",
    "cbm_leaky": "cbm_leaky_5concepts",
    "resnet50": "baseline_resnet50",
    "densenet121": "baseline_densenet121",
    "efficientnet_b4": "baseline_efficientnet_b4",
    "vit_small": "baseline_vit_small",
    "deit_small": "baseline_deit_small",
}
BATCH = {
    "convnext_blackbox": 32,
    "cbm_nonleaky": 32,
    "cbm_leaky": 32,
    "resnet50": 32,
    "densenet121": 32,
    "efficientnet_b4": 16,
    "vit_small": 16,
    "deit_small": 16,
}
ACCUM = {m: max(1, 32 // BATCH[m]) for m in BATCH}

def run(cmd, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, env=env)

def input_roots():
    roots = [WORK]
    if INPUT.exists():
        roots += [p for p in INPUT.iterdir() if p.is_dir()]
        datasets = INPUT / "datasets"
        if datasets.exists():
            for owner in datasets.iterdir():
                if owner.is_dir():
                    roots += [p for p in owner.iterdir() if p.is_dir()]
    return roots

def looks_like_code(p):
    return (
        (p / "scripts" / "train.py").exists()
        and (p / "configs").exists()
        and (p / "scripts" / "feature_map_smoothness.py").exists()
    )

def find_code_source():
    candidates = [
        INPUT / "spinexnet-code",
        INPUT / "spinexnet-code" / "het-spine",
        INPUT / "datasets" / "vasuaashadesai" / "spinexnet-code",
    ]
    for r in input_roots():
        candidates += [r, r / "het-spine", r / "spinexnet-code"]
    for p in candidates:
        if looks_like_code(p):
            return p
    raise FileNotFoundError("Could not find spinexnet-code with scripts/train.py")

def first_existing(candidates, label):
    for p in map(Path, candidates):
        if p.exists():
            return p
    preview = [str(p) for p in candidates[:10]]
    raise FileNotFoundError(f"Could not find {label}. Tried: {preview}")

def find_manifest():
    candidates = [
        INPUT / "manifests-of-spinexnet" / "manifests" / "manifest_v2.csv",
        INPUT / "datasets" / "vasuaashadesai" / "manifests-of-spinexnet" / "manifests" / "manifest_v2.csv",
    ]
    for r in input_roots():
        candidates += [r / "manifest_v2.csv", r / "manifests" / "manifest_v2.csv"]
    return first_existing(candidates, "manifest_v2.csv")

def find_cache():
    candidates = [
        INPUT / "pre-processed-crop-224" / "image_cache_224",
        INPUT / "datasets" / "vasuaashadesai" / "pre-processed-crop-224" / "image_cache_224",
    ]
    for r in input_roots():
        candidates += [r / "image_cache_224", r / "pre-processed-crop-224" / "image_cache_224"]
    candidates = [p for p in candidates if (Path(p) / "images_uint8.npy").exists()]
    return first_existing(candidates, "image_cache_224")

def gpu_count():
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0

def run_jobs(jobs, parallel_if_2gpu=True):
    """Run [(name, cmd), ...]. On multi-GPU Kaggle, run up to one job per GPU."""
    n_gpu = gpu_count()
    if parallel_if_2gpu and n_gpu >= 2 and len(jobs) > 1:
        print(f"Detected {n_gpu} GPUs; running up to {n_gpu} jobs concurrently.", flush=True)
        for start in range(0, len(jobs), n_gpu):
            group = jobs[start:start+n_gpu]
            procs = []
            for local_idx, (name, cmd) in enumerate(group):
                env = os.environ.copy()
                env["CUDA_VISIBLE_DEVICES"] = str(local_idx)
                env["PYTHONUNBUFFERED"] = "1"
                print("\n$", " ".join(map(str, cmd)), f"  # {name} on visible GPU {local_idx}", flush=True)
                procs.append((name, subprocess.Popen(list(map(str, cmd)), env=env)))
            failures = []
            for name, proc in procs:
                rc = proc.wait()
                if rc != 0:
                    failures.append((name, rc))
            if failures:
                raise subprocess.CalledProcessError(failures[0][1], failures[0][0])
    else:
        if len(jobs) > 1:
            print(f"Detected {n_gpu} GPU(s); running jobs sequentially.", flush=True)
        for name, cmd in jobs:
            run(cmd)

def safe_same_path(a, b):
    try:
        return Path(a).resolve() == Path(b).resolve()
    except Exception:
        return False

def merge_previous_outputs():
    """Copy prior per-model Kaggle output datasets into /kaggle/working.

    Attach the previous output dataset for this same notebook. Existing files in
    /kaggle/working win, so reruns can resume safely.
    """
    for tree_name in ["outputs", "eval_multifold", "xai_multifold"]:
        dest = WORK / tree_name
        for root in input_roots():
            src = root / tree_name
            if src.exists() and not safe_same_path(src, dest):
                print(f"Merging prior {tree_name}: {src} -> {dest}", flush=True)
                shutil.copytree(src, dest, dirs_exist_ok=True)
    for pattern in ["classification_metrics_*", "xai_multifold_summary_*"]:
        for root in input_roots():
            for src in root.glob(pattern):
                if src.is_file() and not safe_same_path(src, WORK / src.name):
                    shutil.copy2(src, WORK / src.name)

SRC = find_code_source()
CODE = WORK / "spinexnet-code"
if SRC.resolve() != CODE.resolve():
    shutil.copytree(SRC, CODE, dirs_exist_ok=True)

MANIFEST = find_manifest()
CACHE = find_cache()
CFG = {m: CODE / rel for m, rel in CFG_NAME.items()}
merge_previous_outputs()

def find_checkpoint(model, fold):
    rels = [
        Path("outputs") / EXP[model] / f"fold_{fold}" / "best.pt",
        Path(EXP[model]) / f"fold_{fold}" / "best.pt",
        Path("checkpoints") / f"{model}_fold_{fold}_best.pt",
        Path("checkpoints") / f"{model}_fold{fold}_best.pt",
    ]
    if fold == 0:
        rels.append(Path("checkpoints") / f"{model}_best.pt")
    candidates = []
    for r in input_roots():
        candidates += [r / rel for rel in rels]
    candidates += [WORK / rel for rel in rels]
    return first_existing(candidates, f"{model} fold {fold} checkpoint")

print("MODEL:", SELECTED_MODEL)
print("CODE:", CODE)
print("MANIFEST:", MANIFEST)
print("CACHE:", CACHE)
print("GPUs:", gpu_count())


In [ ]:

model = SELECTED_MODEL
jobs = []
for fold in FOLDS_TO_TRAIN:
    out = WORK / "outputs" / EXP[model] / f"fold_{fold}"
    if (out / "best.pt").exists():
        print("Skip existing:", out / "best.pt")
        continue
    jobs.append((
        f"train {model} fold {fold}",
        [
            PY, CODE / "scripts/train.py",
            "--config", CFG[model],
            "--manifest", MANIFEST,
            "--fold", fold,
            "--cache-dir", CACHE,
            "--output-dir", out,
            "--batch-size", BATCH[model],
            "--grad-accum", ACCUM[model],
            "--time-limit-minutes", 500,
            "--checkpoint-every-minutes", 20,
            "--heartbeat-every-minutes", 5,
            "--no-data-parallel",
        ],
    ))

run_jobs(jobs, parallel_if_2gpu=PARALLEL_FOLDS_IF_2GPU)


In [ ]:

model = SELECTED_MODEL
eval_root = WORK / "eval_multifold"
for fold in EVAL_FOLDS:
    ckpt = find_checkpoint(model, fold)
    out = eval_root / model / f"fold_{fold}"
    if (out / "metrics_val.json").exists():
        print("Skip existing:", out / "metrics_val.json")
        continue
    run([
        PY, CODE / "scripts/evaluate.py",
        "--config", CFG[model],
        "--checkpoint", ckpt,
        "--manifest", MANIFEST,
        "--fold", fold,
        "--cache-dir", CACHE,
        "--output-dir", out,
    ])

rows = []
for path in (eval_root / model).glob("fold_*/metrics_val.json"):
    with open(path) as f:
        d = json.load(f)
    rows.append({"model": model, "fold": path.parent.name, **d})

df = pd.DataFrame(rows)
if not df.empty:
    df["fold_idx"] = df["fold"].str.extract(r"(\d+)").astype(int)
    df = df.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    df.to_csv(WORK / f"classification_metrics_{model}_by_fold.csv", index=False)
    metrics = ["weighted_log_loss", "balanced_accuracy", "macro_f1", "accuracy", "auc_ovr"]
    summary = df.groupby("model")[metrics].agg(["mean", "std"]).round(4)
    summary.to_csv(WORK / f"classification_metrics_{model}_mean_std.csv")
    display(summary)
else:
    print("No evaluation metrics found yet.")


In [ ]:

model = SELECTED_MODEL
jobs = []
for fold in XAI_FOLDS:
    ckpt = find_checkpoint(model, fold)
    out = WORK / "xai_multifold" / f"fold_{fold}" / model
    if (out / "xai_summary_v2.json").exists():
        print("Skip existing:", out / "xai_summary_v2.json")
        continue
    cmd = [
        PY, CODE / "scripts/run_xai_benchmark_v2.py",
        "--config", CFG[model],
        "--checkpoint", ckpt,
        "--manifest", MANIFEST,
        "--fold", fold,
        "--cache-dir", CACHE,
        "--output-dir", out,
        "--max-samples", MAX_SAMPLES,
        "--methods", *METHODS,
        "--skip-consistency",
    ]
    if SAVE_MAPS:
        cmd.append("--save-maps")
    if not RUN_CONSENSUS:
        cmd.append("--skip-consensus")
    if model in {"vit_small", "deit_small"}:
        cmd.append("--enable-attention-rollout")
    jobs.append((f"xai {model} fold {fold}", cmd))

run_jobs(jobs, parallel_if_2gpu=PARALLEL_FOLDS_IF_2GPU)

rows = []
for path in (WORK / "xai_multifold").glob(f"fold_*/{model}/xai_summary_v2.json"):
    with open(path) as f:
        d = json.load(f)["summary"]
    rows.append({
        "fold": path.parents[1].name,
        "model": model,
        "mean_spearman": d.get("mean_spearman"),
        "mean_top20_iou": d.get("mean_top20_iou"),
        "consensus_insertion_auc_mean": d.get("consensus_insertion_auc_mean"),
        "consensus_expert_roi_mean": d.get("consensus_expert_roi_mean"),
    })

xai_summary = pd.DataFrame(rows)
if not xai_summary.empty:
    xai_summary["fold_idx"] = xai_summary["fold"].str.extract(r"(\d+)").astype(int)
    xai_summary = xai_summary.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    xai_summary.to_csv(WORK / f"xai_multifold_summary_{model}.csv", index=False)
    display(xai_summary)
else:
    print("No XAI summaries found yet.")
